#### Integration of Apache Kafka and Spark Streaming

Let's give some (admittedly odd) context to this notebook. An alien entity has entered Portugal and suddenly starts turning citizens into scotsmen! Fear grips the country. We need to find out what is happening and how it is affecting the population.

[To understand what is going on](https://www.youtube.com/watch?v=qxDJMn-534Y)
(This is a Monty **Python** sketch)

We have been charged by the Portuguese government to analyze how the population has already changed. We will use **Spark Streaming** to stream data from our Kafka cluster and analyze live-streamed epidemiological data.

We will work with streams, stream-stream joins and stream-static joins.

Stream-stream joins involve combining two continuous flows of data, which is more complex because events from both sources may arrive at different times or out of order. To make this work, Spark must temporarily store the data from both streams to check for matches, within, for exampke, a 5-minute window. To prevent the system from running out of memory, you must define "watermarks" and time constraints, which tell the engine how long it needs to wait for late data before it can safely discard old records.

Stream-static joins are the simpler of the two join types, used primarily to enrich real-time data with fixed reference information. This allows you to add context to your streaming data without needing to manage complex state for the static side of the join.

<img src="img/scottish_portugal.png" height="500" width="700"/>

Structured Streaming treats a ``stream`` of data as a table that is updated in real time. An underlying process then regularly checks for updates and updates the table, if necessary. The API around Structured Streaming is designed in such a way that what works on your DataFrame, should also work on your streamed DataFrame! 

``Spark Streaming`` is a subset of Spark's functionalities that allows us to work with event-based data, as with our Kafka cluster. We set some global variables and import Schema Types to **structure our data**.

In [ ]:
import pyspark.sql.functions as F

from pyspark.sql.types import StructType, StringType, DoubleType, StructField, IntegerType, TimestampType
from pyspark.sql import SparkSession

KAFKA_BOOTSTRAP_SERVERS = "localhost:8098"
KAFKA_TOPIC = "scotsmen"

We initialize a Spark Session. We import the ``Spark SQL Kafka Connector`` as a dependency. 

In [ ]:
# Initialize local spark session
spark = SparkSession \
    .builder \
    .appName("kafka_streaming") \
    .config("spark.streaming.stopGracefullyOnShutdown", True) \
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0') \
    .config("spark.sql.shuffle.partitions", 4) \
    .master("local[*]") \
    .getOrCreate()

Using the ``subscribe-publish`` paradigm, we subscribe to the Kafka topic ``scotsmen``.

In [ ]:
# TODO: Read from ``KAFKA_TOPIC`` using spark.readStream
# Ensure you configure the bootstrap servers, the subscribe option, and startingOffsets

streaming_df = # ... code here


In [ ]:
# TODO: Instantiate the schema of the messages received
# Fields: district (String), new_scotsmen (Integer), timestamp (Timestamp)

scotsmen_schema = # ... code here

In [ ]:
# TODO: We select the 'value' column and cast it as a String
# Use F.from_json to apply the schema

json_df = # ... code here

In [ ]:
# We instantiate an SQL view to inspect our data
json_df.select("value.*").createOrReplaceTempView("scotsmen")

In [ ]:
# # Sample query from ``scotsmen`` table
# scotsmen_query = spark.sql("SELECT * FROM scotsmen")

# query = scotsmen_query.writeStream.toTable("my_table")

Note that, as with the non-streaming API, there are ``transformations`` and ``actions``. Execution of a query operation on Spark Streaming is lazy.

### Input Sources & Sinks

Spark Structured Streaming supports different input sources and sinks. Supported sinks are:
1. Kafka Streams
2. Files on a distributed file system (HDFS, S3). Spark will read files from a directory
3. A Socket Source

While input sources specify the origin of the data, sinks specify where the data will be written. Those sinks can be:
1. Kafka sink: Pushes data to Kafka
2. Files sink: Writes the output to a file (JSON, parquet, CSV etc.)
3. ForEach sink: Can be used to for each row of a DataFrame for custom storage logic
4. Console sink: Used for testing
5. Memory: Used for debugging

``Memory`` and ``Console``sinks are very similar. ``Memory`` mode makes the data available in an in-memory table for interactive inspection.

In [ ]:
# TODO: Write the results to the console
# Use outputMode "append" and format "console"

query = # ... code here

In [ ]:
# Stopping the query
query.stop()

There are three different output modes available. Here, we used ``append``, which only adds new records to the sink. The other two are ``update`` and ``complete``. ``update`` mode updates the data in the sink, while ``complete`` mode replaces the data in the sink.

In [ ]:
# TODO: Write this query to a memory table called "scotsmen_table"
# Use outputMode "append" and format "memory"

memory_query = # ... code here

In [ ]:
spark.sql("SELECT district, sum(new_scotsmen) AS total_new_scotsmen FROM scotsmen_table GROUP BY district").show()

#### Window Functions

In [ ]:
# Convert JSON to DataFrame
new_scotsmen_df = json_df.select(
    F.col("value.district").alias("conversion_district"),
    F.col("value.timestamp").alias("conversion_timestamp"),
    F.col("value.new_scotsmen").alias("new_scotsmen")
)

In [ ]:
# TODO: Apply Watermarking and Windowing
# 1. Add watermark (30 seconds) on "conversion_timestamp"
# 2. Group by a 3 minute window on "conversion_timestamp" AND "conversion_district"
# 3. Aggregate: count, sum(new_scotsmen), and avg(new_scotsmen)

windowed_df = # ... code here

In [ ]:
window_query = windowed_df \
    .writeStream \
    .outputMode("complete") \
    .queryName("new_scotsmen_aggregated") \
    .format("memory") \
    .start()

In [ ]:
spark.sql("SELECT * FROM new_scotsmen_aggregated").show()

#### Advanced Features

Structured Streaming supports ``Joins``. This means that you are able to (I) join a stream with a static DataFrame and (II) join two streams. This can be used to supplement streaming data with another data source.

Here, we will supplement our ``scotsmen`` table with the ``bag_pipes_sales`` table.

In [ ]:
# Again, we need to read from Kafka.
# This time, we subscribe to the bagpipes topic
bagpipes_stream = spark.readStream.format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS) \
    .option("subscribe", "bagpipe") \
    .option("startingOffsets", "earliest") \
    .load()    

In [ ]:
# Instantiate the schema of the messages received
bagpipes_schema = StructType([
  StructField("district", StringType()),
  StructField("bagpipe_sales", IntegerType()),
  StructField("timestamp", TimestampType())
])

In [ ]:
# We select the 'value' column and cast it as a String
bagpipes_json_df = bagpipes_stream.select(
    F.from_json(F.col("value").cast("string"), bagpipes_schema).alias("value"), 
    )

In [ ]:
# Convert the JSON to DataFrame and alias columns
bagpipe_df = bagpipes_json_df.select(
    F.col("value.district").alias("sales_district"),
    F.col("value.timestamp").alias("sale_timestamp"),
    F.col("value.bagpipe_sales").alias("bagpipe_sales")
)

In [ ]:
# Add watermarking to both streams to handle late data
conversion_stream = new_scotsmen_df.withColumn("conversion_truncated_timestamp", F.date_trunc("minute", new_scotsmen_df["conversion_timestamp"]))
bagpipes_stream = bagpipe_df.withColumn("sales_truncated_timestamp", F.date_trunc("minute", bagpipe_df["sale_timestamp"]))

# Watermark the datasets
conversion_stream = conversion_stream.withWatermark("conversion_truncated_timestamp", "1 minute")
bagpipes_stream = bagpipes_stream.withWatermark("sales_truncated_timestamp", "1 minute")

# Alias the datasets
conversion_stream = conversion_stream.alias("s1")
bagpipes_stream = bagpipes_stream.alias("s2")

# TODO: Perform the join between the two windowed streams
# Join Criteria:
# 1. District matches
# 2. sales timestamp >= conversion timestamp
# 3. sales timestamp <= conversion timestamp + 5 minute interval
joined_stream = # ... code here


# Create the window column before aggregation
joined_stream = joined_stream.withColumn("window", F.window("sales_truncated_timestamp", "2 minutes"))

# Aggregate values per district
aggregated_stream = joined_stream \
    .groupBy(
        joined_stream.conversion_district,
        joined_stream.window
    ) \
    .agg(
        F.sum(joined_stream.new_scotsmen).alias("total_new_scotsmen"),
        F.sum(joined_stream.bagpipe_sales).alias("total_bagpipe_sales")
    )

In [ ]:
# Output the results to the console for inspection
query = aggregated_stream \
    .writeStream \
    .queryName("stream_stream_join") \
    .outputMode("append") \
    .format("memory") \
    .start()

In [ ]:
# Showing the results
spark.sql("SELECT * FROM stream_stream_join").show()

#### Static-Stream Joins

Apart from joining two streams, Spark also supports joining a stream with a static DataFrame. This can be used to supplement streaming data with another data source, such as a lookup table. Here  we will supplement our ``conversation_stream`` with the ``portugal_district_population2022.csv`` table.

In [ ]:
population_schema = StructType([
  StructField("district", StringType()),
  StructField("pop", IntegerType())
])

population_df = spark \
    .read \
    .format("csv") \
    .option("header", True) \
    .schema(population_schema) \
    .load("portugal_district_population2022.csv")

In [ ]:
# TODO: Join the conversion_stream with the static population_df
# Match on district columns
joined_stream_population = # ... code here

# Calculate conversions per population
joined_stream_population = joined_stream_population \
    .withColumn(
        "conversions_per_pop", F.col("new_scotsmen") / F.col("pop")
    )

# Select the columns you need
result_stream = joined_stream_population.select(
    "conversion_truncated_timestamp",
    "new_scotsmen",
    "conversion_district",
    "pop",
    "conversions_per_pop"
)

In [ ]:
# Output the results to the console for testing
query = result_stream \
    .writeStream \
    .queryName("prop_pop_converted") \
    .outputMode("append") \
    .format("memory") \
    .start()

In [ ]:
# Define the window duration and slide duration
window_duration = "1 hour"
slide_duration = "10 minutes"

# SQL query to compute the cumulative sum of the ratio
sql_query = f"""
SELECT
    window.start AS window_start,
    window.end AS window_end,
    SUM(conversions_per_pop) AS cumulative_ratio,
    conversion_district
FROM (
    SELECT
        conversions_per_pop,
        window(current_timestamp(), '{window_duration}', '{slide_duration}') AS window,
        conversion_district
    FROM prop_pop_converted
)
GROUP BY conversion_district, window
ORDER BY window_start
"""

# Execute the SQL query
result_df = spark.sql(sql_query)

# Show the result
result_df.show()

#### Simulating Streaming Datasets

It is also possible to "simulate" a streaming dataset by reading from a directory of CSV files. This can be useful for testing and debugging purposes. The dataset contains individual files that can be read as a batch of data.

In [ ]:
# Define the weather schema
schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("min_temperature", DoubleType(), True),
    StructField("max_temperature", DoubleType(), True),
    StructField("precipitation", DoubleType(), True)
])

# Path to the directory containing the CSV files
# Note that this must be a directory and not an individual file
input_path = "weather"

NUM_FILES_PER_TRIGGER = 3

# TODO: Read the streaming DataFrame from the directory
# Use spark.readStream, format "csv", and option "maxFilesPerTrigger"
streaming_df = # ... code here

# TODO: Define the query to process the streaming data
# Use processingTime trigger of 10 seconds
query = # ... code here

In [ ]:
# We can now query the static dataset just like we did before
spark.sql(
    """SELECT max_temperature, min_temperature, timestamp 
    FROM weather ORDER BY timestamp 
    DESC
    """
).show()